In [2]:
import pandas as pd
import json
import ollama
from tqdm import tqdm
import time
import os

#load data
train_df = pd.read_csv('../data/raw/MTS-Dialog-TrainingSet.csv')
print(f"Dialogues to process: {len(train_df)}")
print(f"Median dialogue length: {train_df['dialogue'].str.len().median():.0f} characters")

#check if we have results from a previous run
output_path = '../data/processed/mts_entities.json'
if os.path.exists(output_path):
    with open(output_path, 'r') as f:
        existing = json.load(f)
    print(f"Found {len(existing)} existing extractions")
else:
    existing = {}
    print("starting fresh")

Dialogues to process: 1201
Median dialogue length: 361 characters
starting fresh


In [5]:
def extract_entities(dialogue):
    """Extract clinical entities from a dialogue using Ollama."""
    prompt = f"""You are a clinical documentation assistant. Read this doctor-patient dialogue and extract the following as JSON only. No other text, no markdown, no code blocks.

{{
    "chief_complaint": "main reason for visit in a few words",
    "symptoms": ["list of symptoms mentioned"],
    "diagnoses": ["list of diagnoses or conditions discussed"],
    "medications": ["list of medications mentioned"],
    "severity": "low or moderate or high",
    "follow_up_needed": true or false
}}

DIALOGUE:
{dialogue}

JSON:"""

    try:
        response = ollama.generate(model='llama3.1:8b', prompt=prompt)
        text = response['response'].strip()
        if '```json' in text:
            text = text.split('```json')[1].split('```')[0]
        elif '```' in text:
            text = text.split('```')[1].split('```')[0]
        return json.loads(text)
    except Exception as e:
        return None

#run extraction
results = existing.copy()
failed = []
start_time = time.time()

print(f"Processing {len(train_df)} dialogues...\n")
for i, row in tqdm(train_df.iterrows(), total=len(train_df)):
    str_id = str(row['ID'])

    #skip if already processed
    if str_id in results:
        continue

    result = extract_entities(row['dialogue'])
    if result:
        results[str_id] = result
    else:
        failed.append(str_id)

    #save every 50
    if len(results) % 50 == 0:
        with open(output_path, 'w') as f:
            json.dump(results, f)

#last save
with open(output_path, 'w') as f:
    json.dump(results, f)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Successful: {len(results)}/{len(train_df)}")
print(f"Failed: {len(failed)}")

Processing 1201 dialogues...



100%|██████████| 1201/1201 [35:38<00:00,  1.78s/it]


Done in 35.6 minutes
Successful: 1200/1201
Failed: 1


In [ ]:
# Load the extracted entities
with open('../data/processed/mts_entities.json', 'r') as f:
    all_entities = json.load(f)

print(f"Loaded entities for {len(all_entities)} dialogues")

# Look at a few examples
for id_key in list(all_entities.keys())[:3]:
    print(f"\nID {id_key}:")
    print(json.dumps(all_entities[id_key], indent=2))